# Podcast Activity Tracker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/apple-podcasts-activity-tracker.ipynb)

**What:** Find active Apple Podcasts shows by keyword, chart or genre. Each row adds real activity from the show RSS feed: last episode, cadence, episodes per 90 days and a 0-100 activity score. No emails, no login, no API key.

**Why:** run the [`apple-podcasts-activity-tracker`](https://apify.com/transparent_meteorite/apple-podcasts-activity-tracker) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

## 1. Setup

In [ ]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/apple-podcasts-activity-tracker) for every option.

In [ ]:
run_input = {'searchTerms': ['marketing'], 'countries': ['us'], 'onlyActive': True, 'maxItems': 10}

## 3. Run the actor and read the dataset

In [ ]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/apple-podcasts-activity-tracker').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

## 4. Tabulate

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["title", "publisher", "activityScore", "isActive", "lastEpisodeAt", "daysSinceLastEpisode", "episodesLast90Days", "primaryGenre", "country"] if c in df.columns]
df[SHOW]

## 5. A quick analysis

In [ ]:
for c in ['activityScore','episodesLast90Days','daysSinceLastEpisode']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
print('Active shows:',int(df['isActive'].sum()),'of',len(df))
df.sort_values('activityScore',ascending=False)[['title','activityScore','episodesLast90Days']].head(5)

## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [apple-podcasts-activity-tracker on the Apify Store](https://apify.com/transparent_meteorite/apple-podcasts-activity-tracker).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.